In [0]:
%sql
SELECT 
    churn,
    COUNT(*) AS total_clientes,
    -- Percentual em relação ao total de 14.400 churns do banco (se churn = 1)
    ROUND(SUM(CASE WHEN churn = 1 THEN 1 ELSE 0 END) * 100.0 / 14400, 2) AS pct_sobre_total_churns_14400,
    -- Percentual do grupo dentro do próprio segmento Mass
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_do_segmento_mass,
    ROUND(AVG(saldo), 2) AS media_saldo,
    ROUND(AVG(renda_mensal), 2) AS media_renda,
    ROUND(AVG(num_servicos), 2) AS media_servicos,
    ROUND(AVG(score_credito), 2) AS media_score
FROM workspace.default.bank_churn_dataset_pt
WHERE segmento_cliente = 'Mass'
GROUP BY churn;

In [0]:
%sql
SELECT 
    churn,
    membro_ativo,
    COUNT(*) AS total_clientes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY churn), 2) AS percentual_dentro_do_churn
FROM workspace.default.bank_churn_dataset_pt
WHERE segmento_cliente = 'Mass'
GROUP BY churn, membro_ativo
ORDER BY churn, membro_ativo;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    segmento_cliente,
    churn,
    membro_ativo,
    COUNT(*) AS total_clientes,
    (COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY segmento_cliente)) AS percentual_dentro_do_churn
FROM workspace.default.bank_churn_dataset_pt
WHERE segmento_cliente = 'Mass'
GROUP BY churn, membro_ativo, segmento_cliente
ORDER BY churn, membro_ativo;

A Raiz do Churn Descoberta: 90,95% de todos os clientes do segmento Mass que dão churn são classificados como Inativos. Apenas 9,05% dos que cancelam possuíam o status de membro ativo.

O Alerta nos Ativos: Mesmo entre quem não cancelou (churn = 0), a maioria já é inativa (69,62%). Isso mostra que a base de massa sofre de uma enorme "dor de dormência": as pessoas abrem contas, param de usar e, eventualmente, fecham o cadastro de vez.

Plano de Ação Estratégico

Revisão do Onboarding: Como o volume de inativos é alto até mesmo em quem fica (quase 70%), o processo de ativação inicial pós-abertura de conta precisa ser redesenhado para garantir que o cliente realize pelo menos uma transação nos primeiros dias de relacionamento.

In [0]:
%sql
SELECT 
    nivel_fidelidade,
    COUNT(*) AS total_clientes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentual_da_base
FROM workspace.default.bank_churn_dataset_pt
WHERE segmento_cliente = 'Mass'
GROUP BY nivel_fidelidade
ORDER BY total_clientes DESC;

O Peso do Segmento Mass no Nível Bronze:

"Para coroar essa leitura, quando isolamos o segmento Mass, a concentração atinge o seu ápice: 91,09% de toda essa base (19.526 clientes) está estacionada no nível Bronze, enquanto os níveis Silver e Gold representam apenas 6,74% e 2,17%, respectivamente.

Isso prova que o nosso grande volume de prospecção em massa desagua integralmente na base Bronze. O desafio não está em atrair esse público — que chega qualificado em termos de risco —, mas em criar uma esteira de valor ágil que impeça que esses 19 mil clientes Mass fiquem estagnados e inativos, tornando-se o principal combustível da nossa evasão."